# Prediction of overall survival of patients with Myeloid Leukemia

## Importing Modules

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Importing Data

We are importing the datasets and setting the index to 'ID' as it is the unique identifier for each patient.

### 1.1 Clinical dataset

In [ ]:
X_clinical = pd.read_csv('../data/raw/X_train/clinical_train.csv')
X_clinical = X_clinical.set_index('ID')
X_clinical.head()

In [ ]:
print(f"Shape of the clinical dataset: {X_clinical.shape}")
X_clinical.dtypes

Features:
- ID: A unique identifier for each patient.
- CENTER: The clinical center where the patient is being treated.
- BM_BLAST: Percentage of blasts in the bone marrow, indicating the proportion of abnormal blood cells found in patients' bone marrow.
- WBC: White blood cell count, measured in gig/L (blood parameter).
- ANC: Absolute neutrophil count in the blood, in gig/L (blood parameter).
- MONOCYTES: Number of monocytes in the blood, in gig/L (blood parameter).
- HB: Hemoglobin level, measured in g/dL (blood parameter).
- PLT: Platelet count, in gig/L (blood parameter).
- CYTOGENETICS: Detailed description of the patient's karyotype, showing the chromosomal abnormalities detected in the blood cancer cells. (ISCN standard)

### 1.2 Molecular dataset

The molecular dataset is out of scope of this EDA for now; we only load it to check ID coherence

In [ ]:
X_molecular = pd.read_csv('../data/raw/X_train/molecular_train.csv')
X_molecular = X_molecular.set_index('ID')
X_molecular.head()

In [ ]:
print(f"Shape of the molecular dataset: {X_molecular.shape}")
X_molecular.dtypes

### 1.3 Labels dataset

In [ ]:
Y = pd.read_csv('../data/raw/target_train.csv')
Y = Y.set_index('ID')
Y.head()

In [ ]:
print(f"Shape of the labels dataset: {Y.shape}")
Y.dtypes

Labels:
- OS_YEARS: Overall survival time in years from diagnosis.
- OS_STATUS: Survival status indicator, where 1 indicates death and 0 indicates that the patient was alive at the time of the last follow-up. \
\
This prediction problem is a survival analysis problem, not a binary classification problem.

### 1.4 ID coherence

In [ ]:
unique_id = X_clinical.index.is_unique, X_molecular.index.is_unique, Y.index.is_unique
print(f"The ID is unique: X_clinical=  {unique_id[0]}, X_molecular= {unique_id[1]}, Y= {unique_id[2]}")
print(f"Number of ID in Y and not in X_clinical: {len(Y.index.difference(X_clinical.index))}")
print( f"Number of ID in X_clinical and not in Y: {len(X_clinical.index.difference(Y.index))}")
print(f"Number of ID in Y and not in X_molecular: {len(Y.index.difference(X_molecular.index))}")
print(f"Number of ID in X_molecular and not in Y : {len(X_molecular.index.difference(Y.index))}")
print(f"Are all IDs in the molecular dataset in the clinical dataset: {X_molecular.index.isin(X_clinical.index).all()}")

ID coherence in the datasets:
- In the clinical and labels datasets, IDs are unique.
- In the molecular dataset, IDs are not unique, a patient can have several rows.
- All patients in the labels dataset have a record in the clinical datasets and vice versa.
- Some patients have no record in the molecular datasets, but all of the patients in the molecular dataset have a record in the labels dataset and in the clinical dataset.\
\
**A patient can have none, one or multiple mutations.**



## 2. Labels: OS_YEARS & OS_STATUS

### 2.1 Missing data in the labels

Calculating the proportion of missing values.

In [ ]:
Y_missing = pd.DataFrame({'Number of missing data' : Y.isna().sum(), 'Proportion of missing data (%)' : (Y.isna().sum()/Y.shape[0])*100})
Y_missing

In [ ]:
Y_missing_matrix = pd.crosstab(Y['OS_YEARS'].isna(), Y['OS_STATUS'].isna())
Y_missing_matrix

- 150 values are missing in both columns.
- All missing values are missing in both columns simultaneously.
- Missing values account for 4.5% of the dataset. \
\
**We will have to drop these patients, as they would be unusable for the training.**

Let's study the proportion of missing values per center to identify if the distribution is random or not.

In [ ]:
Y_missing_per_center = pd.crosstab(X_clinical['CENTER'], Y['OS_STATUS'].isna(), normalize= 'index')
Y_missing_per_center

The missing data seems to be concentrated in some centers (HMS (38.6%), VU (12%), CCH (11.9%), KI (8.9%),  ...). Dropping the patients will introduce a bias.

In [ ]:
Yv = Y.dropna()
print(f"Shape of Yv: {Yv.shape}")
Yv

### 2.2 Values and coherence between OS_YEARS and OS_STATUS

1. OS_STATUS takes 0 or 1
2. OS_YEARS should be positive
3. Patients who most likely did not have a follow-up or were included at the very end of the study: OS_YEARS = 0 and OS_STATUS = 0
4. Distribution of survival duration per status

In [ ]:
# 1. OS_STATUS takes 0 or 1
print(Y['OS_STATUS'].value_counts(dropna=False))

In [ ]:
# 2. OS_YEARS < 0
print(f"Number of negative values: {(Y['OS_YEARS'] < 0).sum()}")
Y['OS_YEARS'].describe()

In [ ]:
# 3. Patient not under follow-up
print(Y[Y['OS_YEARS'] == 0]['OS_STATUS'].value_counts())

These patients do not bring any information about the survivability. We may have to drop them.

In [ ]:
# 4. Distribution of the survivability
sns.histplot(data=Y, x='OS_YEARS', hue='OS_STATUS', bins=44, multiple='stack')
Y.groupby('OS_STATUS')['OS_YEARS'].describe()

The median of the follow-up for the censored patients (2.26 years) is higher than the median of the follow-up for the deceased patients (1.38 years). This is consistent, patients who survived have a longer follow-up.

### 2.3 Censoring

The data are right-censored. \
OS_YEARS represents the duration of the follow-up. A patient can be included at any point during the ~22 year study period. OS_STATUS indicates whether the death has or has not been observed during the last follow-up. However, we do not know if the follow-up has ended because the study has ended or for another reason (e.g., the follow-up is done somewhere else, ...).

1. We are studying the censoring rate.
2. We are studying the distribution of the censoring in the data.

In [ ]:
# 1. Censoring rate (excluding na)
Y_no_na = Y['OS_STATUS'].notna()
censoring_rate = (Y.loc[Y_no_na, 'OS_STATUS'] == 0).mean()
print(f"Censoring rate (n={Y_no_na.sum()}): {censoring_rate:.1%}")

In [ ]:
# 2. Distribution of the censoring (excluding na)
censoring_distribution = Y.dropna().copy()
censoring_distribution['bin'] = pd.cut(censoring_distribution['OS_YEARS'], bins = [0, 0.5, 1, 1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5, 5.5, 6, 6.5, 7, 7.5, 8, 9, 10, 25], include_lowest=True)
summary = censoring_distribution.groupby('bin', observed=True)['OS_STATUS'].agg(n='count', deceased=lambda d: (d==1).sum(), censored=lambda c: (c==0).sum(), censoring_rate=lambda s: (s == 0).mean().round(2)*100)
summary

In [ ]:
sns.histplot(Y.dropna(), x=Y.loc[Y_no_na,'OS_YEARS'], hue=Y.loc[Y_no_na, 'OS_STATUS'], bins=44)

We do not have the date of inclusion, therefore it is hard to tell if the censoring is due to the end of the study or if the patients stopped participating in the study.
- Global censoring rate  ((among patients with observed labels, n = 3,173): 49.6%.
- The number of deceased is steadily decreasing, the lethality of the disease is high at the beginning.
- The number of censored patients is very high for the 0-6 months (338 patients), the follow-up has ended quickly after the inclusion. This may suggest that a large group of patients was recruited during the final 6 months of the study, that some patients ended the follow-up for another reason or that the duration of the study / the follow-up was not the same for all centers. (This includes the 53 patients with a follow-up duration of 0)
- The censoring rate decreases at the beginning (first 2 years) but steadily increases after. A local dip is observed around 6.5 - 7.5 years but it rests on small bins and may be sampling noise. This is consistent, patients observed for long duration are those who survived the early high-mortality phase. Therefore, most of them reach the end of the study and are censored.

## 3. Clinical data

In this section we are studying the missing data, distribution, outliers then the discete features CENTERS and CYTOGENETICS.

### 3.1 Missing data

We are studying the missing data by feature.

In [ ]:
X_clinic_miss_by_feat = pd.DataFrame({'number of missing data': X_clinical.isna().sum(), 'Proportion of missing data (%)': (X_clinical.isna().sum() / X_clinical.shape[0]) * 100})
X_clinic_miss_by_feat


Some features have a lot of missing values (e.g. MONOCYTES). \
We are studying the missing data by CENTER to identify if these missing data are related to the CENTER where the patient was followed.

In [ ]:
X_clinic_miss_by_center = X_clinical.groupby('CENTER')[X_clinical.columns.drop(['CENTER'])].apply(lambda g: g.isna().mean() * 100)
X_clinic_miss_by_center

In [ ]:
sns.heatmap(X_clinic_miss_by_center, annot=True)

- There is a lot of missing values for the MONOCYTES (18.1%) CYTOGENETICS (11.6%) and WBC (8.2%) features.
- The REL and UOXF centers did not collect any MONOCYTES data. We won't be able to replace these missing values with values estimated per center.
- Missing data are over-represented in some centers: HMS & DUTH.
- Missing data in the MONOCYTES, WBC, ANC and PLT features seem to vary by CENTER
- Missing data in the BM_BLAST and HB features seem to be more randomly distributed \
\
**In the clinical dataset, missing values are not randomly distributed. We must take this into account when replacing missing data.**

### 3.2 Distribution

#### 3.2.1 Hematological parameters distribution

In [ ]:
X_clinical.describe()

In [ ]:
hemo_var = ['BM_BLAST', 'WBC', 'ANC', 'MONOCYTES', 'HB', 'PLT'] # hematological parameters

fig, axes = plt.subplots(2,3, figsize= (20, 10))
axes = axes.flatten()

# Distribution of hematological parameters
for var, ax in zip(hemo_var, axes):
    sns.histplot(data=X_clinical, x=var, bins=30, kde=True, ax=ax)
    ax.set_title(var)

Distribution:
- BM_BLAST, WBC, ANC, MONOCYTES and PLT are right-skewed.
- HB is roughly symmetrical.
- MONOCYTES, ANC and WBC are very concentrated on the low values.
- PLT and BM_BLAST are less concentrated. \
\
This distribution justifies the use of a log-transform. ANC MONOCYTES and BM_BLAST can take 0.

#### 3.2.2 Hematological parameters distribution per OS_STATUS

In [ ]:
df_status = X_clinical.join(Y['OS_STATUS'], how='inner')   # alignment by ID, explicit
df_status = df_status[df_status['OS_STATUS'].notna()]

fig, axes = plt.subplots(2,3, figsize= (20, 10))
axes = axes.flatten()

# Distribution of hematological parameters per status
for var, ax in zip(hemo_var, axes):
    sns.histplot(data=df_status, x=var, hue='OS_STATUS', bins=30, kde=True, ax=ax)
    ax.set_title(var)


*OS_STATUS groups are balanced (see 2.2).*
- For BM_BLAST, the non-deceased patients are concentrated in the lowest category.
- For ANC, WBC and MONOCYTES there is no obvious distribution difference between deceased and censored.
- For PLT deceased patients are concentrated in the lower categories.
- For HB deceased patients seems to be more concentrated in the lower values and the non-deceased seems to be more concentrated in the higher values.


#### 3.2.3 Outliers

Some patients have very high values for WBC, MONOCYTES, ANC and BM_BLAST but the median of those parameters are small. The data are concentrated in the lef side. \
We are studying the outliers to see if they are corresponding to a certain type of population or if they are corresponding to measurement mistakes. \
\
Let's see what proportion of the samples lies in the tail.
To do so we are using the 1,5*IQR criteria.

In [ ]:
tail_rate ={}
for var in hemo_var:
    q1, q3 = X_clinical[var].quantile([.25, .75])
    iqr = q3 - q1
    out = X_clinical[(X_clinical[var] > q3 + 1.5*iqr) | (X_clinical[var] < q1 - 1.5*iqr)] # outliers
    tail_rate[var] = (len(out) / len(X_clinical) * 100)
pd.DataFrame([tail_rate])

More than 3% of the effective are outliers for BM_BLAST, WBC, ANC, PLT and MONOCYTES. These values do not appear to be an error but likely describe subgroups of patient. Even if they are very high compared to the median.

Applying a log scale to see if it will reduce the number of outliers. \
We are drawing box plots with log scale.

In [ ]:
X_clinical_log = np.log1p(X_clinical[hemo_var])

In [ ]:
tail_rate_log ={}
for var in hemo_var:
    q1, q3 = X_clinical_log[var].quantile([.25, .75])
    iqr = q3 - q1
    out = X_clinical_log[(X_clinical_log[var] > q3 + 1.5*iqr) | (X_clinical_log[var] < q1 - 1.5*iqr)] # outliers
    tail_rate_log[var] = (len(out) / len(X_clinical_log) * 100)
pd.DataFrame([tail_rate_log])

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 10))
axes = axes.flatten()

for var, ax in zip(hemo_var, axes):
    sns.boxplot(data=X_clinical, x=var, ax=ax)
    ax.set_title(var)
    ax.set_xscale('log')

The log1p transformation solves the tail problem for BM_BLAST, ANC and PLT. But the tail is still heavy for the MONOCYTES and WBC. MONOCYTES and WBC keep extreme subgroups even in log scale.

Let's study the proportion of WBC values in the tail by CENTERS.

In [ ]:
wbc_out = X_clinical
q1, q3 = X_clinical['WBC'].quantile([.25, .75])
iqr = q3 - q1
print(q3 + 1.5*iqr)
wbc_out['WBC_out'] = (wbc_out['WBC'] > q3 + 1.5*iqr)
wbc_out.groupby('CENTER').agg(wbc_out_pct=('WBC_out', 'mean')).sort_values('wbc_out_pct', ascending=False)

Some centers are concentrating the higher values for WBC but none sems to really concentrated all the tail value, this rules out a measurement error specific to one center.

Let's study the difference between the median for all patients and the median of patients with high value for WBC (the hyper group). We are taking an arbitrary threshold of 50 for WBC.

In [ ]:
hemo_vars = ['BM_BLAST', 'WBC', 'ANC', 'MONOCYTES', 'HB', 'PLT']
df_plot = X_clinical
df_plot['HYPER'] = df_plot['WBC'] > 50

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for var, ax in zip(hemo_vars, axes):
    sns.boxplot(data=df_plot, x='HYPER', y=var, ax=ax)
    ax.set_yscale('log')
    ax.set_title(var)

The outliers for WBC have a high median for ANC and MONOCYTES. This suggests that outliers are subgroups of patients in our datasets.

We have high proportion of outlier, and the log-transformation does not normalize the distribution. Moreover, the study of outliers for WBC show that they appers to belong to the outliers of other features. This suggests that outliers may correspond to a subgroup of patients

### 3.3 Discrete features

#### 3.3.1 CENTER

In [ ]:
print(f"Number of missing data in centers: {X_clinical['CENTER'].isna().sum()}")
print(f"Number of different centers: {X_clinical['CENTER'].nunique()}")
center_counts = X_clinical['CENTER'].value_counts()
center_counts

We have 23 different centers. The number of patients per center varies widely ranging 6 to 900.

Crossing CENTER with OS_STATUS.

In [ ]:
center_x_os_status = pd.crosstab(X_clinical['CENTER'], Y['OS_STATUS'], normalize='index')
center_x_os_status

We are calculating
- the deceased rate and the censoring rate per center.
- the gap between the deceased rate for this center and the global deceased rate.
- the standard deviation (Sigma). \
Centers with a gap > 2*sigma are centers where a deceased rate variation compared to the global deceased rate can't be explained by the noise, it is a real difference.

In [ ]:
temp = X_clinical.join(Y)
print(temp.shape)
deceased_per_center = temp.groupby('CENTER')['OS_STATUS'].agg(n= 'count', Deceased= 'sum')
# Adding Deceased rate
deceased_per_center['Deceased rate'] = deceased_per_center['Deceased']/deceased_per_center['n']
# Adding Censoring rate
deceased_per_center['Censoring rate'] = 1 - deceased_per_center['Deceased rate']
# Adding the gap between the center deceased rate and the global deceased rate
p_global = temp['OS_STATUS'].mean()
deceased_per_center['Gap'] = (deceased_per_center['Deceased rate'] - p_global).abs()
# Adding standard deviation per center
deceased_per_center['Sigma'] = (p_global * (1 - p_global) / deceased_per_center['n']) ** 0.5
# Comparing the gap to the standard deviation
deceased_per_center['Real difference'] = deceased_per_center['Gap'] > 2 * deceased_per_center['Sigma']

deceased_per_center

The deceased rate per center is heterogeneous. \
For the center with a real difference, we have two categories :
 - the ones with high deceased rate and low censoring rate
 - the ones with low deceased rate and high censoring rate \
\
But we cannot determine if the high censoring rate is due to a short follow-up (the patient is still alive because the follow-up started at the end of the study) or if the follow-up was longer and the patient survived.

Let's calculate the median of the follow-up duration, for the censored patients, per center.

In [ ]:
deceased_per_center['Follow_up_median'] = temp[temp['OS_STATUS'] == 0].groupby('CENTER')['OS_YEARS'].agg(follow_up_median='median') # median of OS_YEARS for censored patients per center

deceased_per_center[['Follow_up_median', 'Censoring rate']]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(deceased_per_center['Censoring rate'], deceased_per_center['Follow_up_median'],
           s=deceased_per_center['n'] * 3, alpha=0.5, edgecolor='black')

for center, row in deceased_per_center.iterrows():
    ax.annotate(center, (row['Censoring rate'], row['Follow_up_median']),
                fontsize=8, xytext=(4, 4), textcoords='offset points')

ax.set_xlabel('Censoring rate')
ax.set_ylabel('Follow-up median (years)')
ax.set_title('Censoring vs follow-up median per center (size = sample size)')

The higher the censoring rate is, the shorter the follow-up median is. The high censoring rate in some center is due to a short follow-up in this center. A center with long follow-up observe mor death. This support the multi-center censoring hypothesis raised in section 2.3.

#### 3.3.2 CYTOGENETICS

In [ ]:
print(f"Number of missing data in cytogenetics: {X_clinical['CYTOGENETICS'].isna().sum()}")
print(f"Number of unique karyotype: {X_clinical['CYTOGENETICS'].nunique()}")

There are 387 missing values in CYTOGENETICS. There are 1,194 unique karyotypes.

Let's examine what are the possible values for CYTOGENETICS.

In [ ]:
cyto_counts = X_clinical['CYTOGENETICS'].value_counts()
cyto_counts.head(15)

In [ ]:
print(f"Number of karyotype seen one time: {(cyto_counts == 1).sum()}")
print(f"Number of patients who have the top 15 karyotype: {cyto_counts.head(15).sum()}")

1,075 karyotypes out of 1,194 are only seen in one patient. 1,428 patients have one of the top 15 most frequent karyotype in the dataset.

Let's study the lengths of the karyotype.

In [ ]:
X_clinical['CYTOGENETICS'].dropna().str.len().describe()

In [ ]:
X_clinical['CYTOGENETICS'].dropna()[X_clinical['CYTOGENETICS'].dropna().str.len() > 30].head(10)

The median is 9 characters, more than 50% of the dataset are simple karyotype.

In [ ]:
X_clinical['CYTOGENETICS'].head()

We have several non-standard karyotypes, some entries simply state 'complex' or 'abnormal' instead of a full karyotype. Some karyotypes do not specify the anomaly but the number of chromosomes is abnormal.

Let's build a filter to select the normal karyotypes. \
We are describing what a normal karyotype is. \
A normal karyotype is 46 followed by xx or xy and an optional [number].

In [ ]:
is_normal = X_clinical['CYTOGENETICS'].str.match(r'46,(xx|xy)(\[\d+\])?$', case=False)
X_clinical[is_normal == True]['CYTOGENETICS']

We are crossing the karyotype status with the survivability status. We are excluding the missing values for the labels.

In [ ]:
pd.crosstab(is_normal, Y['OS_STATUS'].dropna(), normalize='index', margins=True)

In [ ]:
df_karyo = X_clinical[['CYTOGENETICS']].join(Y, how='inner')
# Creation of the normal/abnormal feature (specifying if missing values)
df_karyo['KARYOTYPE'] = np.where(df_karyo['CYTOGENETICS'].isna(), 'MISSING',np.where(is_normal, 'NORMAL', 'ABNORMAL'))
# Calculating the effective and number of deceased per category
summary = (df_karyo.loc[df_karyo['OS_STATUS'].notna()].groupby('KARYOTYPE')['OS_STATUS'].agg(n='count', deaths='sum'))
# Calculating the death_rate
summary['death_rate'] = summary['deaths'] / summary['n']
# Proportion of deceased in the dataset
p_global = df_karyo.loc[df_karyo['OS_STATUS'].notna(), 'OS_STATUS'].mean()
# Calculating the expected standard deviation of the category
summary['sigma'] = np.sqrt(p_global * (1 - p_global) / summary['n'])
# Comparing the difference to the expected standard deviation
summary['real_diff'] = (summary['death_rate'] - p_global).abs() > 2 * summary['sigma']
summary

The deceased rate is substantially higher for the abnormal (59%) karyotype than for the normal one (42%). This test does not account for censoring and Kaplan-Meier comparison per group would be the proper way to check this hypothesis. Patients with missing data for the karyotype have a substantially higher death rate which suggests a bias in the missing data.

## Conclusion

**Findings:** \
The cohort contains 3,323 patients, 3,173 of whom (95.5%) have observed survival labels; the 150 patients missing both OS_YEARS and OS_STATUS are unusable for survival modeling. The outcome is right-censored, with a censoring rate of 49.6%. Deaths are concentrated within the first year (median time to death: 1.4 years), consistent with the high early mortality of ML. Censoring shows the signature of administrative, multi-center censoring: the largest group of censored patients (338, including 53 with zero follow-up) occurs within the first 6 months, and centers with short follow-up medians (FLO, PV, VU) are also those with the highest censoring rates (Section 3.3.1). Hematological parameters are right-skewed except HB, with heavy tails that represent clinically plausible subpopulations rather than measurement errors. Missing data are not randomly distributed: MONOCYTES (18.1%), CYTOGENETICS (11.6%) and WBC (8.2%) are missing in a center-dependent manner, and patients with missing cytogenetics have a significantly higher crude death rate (58% vs 50.4%), close to the abnormal-karyotype group (58.9% vs 42.1% for normal karyotypes).

**Implications for preprocessing and modeling:** \
First, this is a survival-analysis problem: models and metrics must handle censoring (e.g., Cox regression, AFT models, or Random Survival Forests, evaluated with the concordance index), under the assumption of independent censoring — supported by its administrative pattern. Second, right-skewed variables should be log1p-transformed (zeros are present), and derived ratios such as ANC/WBC should be considered, since WBC, ANC and MONOCYTES are biologically dependent. Third, imputation must account for the center structure — within-center imputation is impossible for MONOCYTES in centers like REL and UOXF — and missingness indicators should be added, since missing cytogenetics is itself prognostic and must be treated as its own category rather than imputed. Fourth, a NORMAL_KARYOTYPE flag should be derived from the raw cytogenetics string. Finally, the 53 patients with zero follow-up carry no survival information and should be handled explicitly.